# T2.2 Semantic Mapping

Owner: B

This notebook maps every column in the DBRepo tables to a concept in a recognised ontology.
This addresses the F (Findable) and I (Interoperable) aspects of FAIR.

## Ontology selection and justification

We use **DBpedia** as the primary ontology for all column mappings. DBpedia is a widely used linked data ontology that extracts structured information from Wikipedia and is one of the most interconnected datasets in the Linked Open Data cloud, providing stable URIs for a wide range of concepts including chemical elements, physical properties, and geographic attributes.

Our preferred domain-specific ontologies would have been SOSA/SSN (for observation structure, station identity, and quality flags), CHEBI (for chemical species such as Pb, Cd, Ca, SO4), and ENVO (for precipitation as an environmental phenomenon), as these are more precise and formally curated for environmental sensor data. However, the DBRepo test instance only accepts URIs from pre-registered ontology namespaces, and these ontologies are not registered there. We contacted the lecture team, who confirmed this limitation and advised us to use the available ontologies instead. A formal request was submitted to register SOSA, CHEBI, ENVO, and PATO in the test instance.

DBpedia was used as the fallback because it is one of the registered namespaces in DBRepo and provides the closest available concepts for our dataset. WikiData was initially considered but caused HTTP 500 errors on the DBRepo test instance and was therefore not used.

## Full mapping table

### Table: stations

| Column | Concept | URI |
|--------|---------|-----|
| station_code | location | http://dbpedia.org/ontology/location |
| latitude | latitude | http://dbpedia.org/ontology/latitude |
| longitude | longitude | http://dbpedia.org/ontology/longitude |

### Table: precipitation_measurements

| Column | Concept | URI |
|--------|---------|-----|
| sample_date | date | http://dbpedia.org/ontology/date |
| station_id | location | http://dbpedia.org/ontology/location |
| NS | precipitation | http://dbpedia.org/ontology/precipitation |
| LF | electricalConductivity | http://dbpedia.org/ontology/electricalConductivity |
| pH | pH | http://dbpedia.org/ontology/pH |
| NH4 | Ammonium | http://dbpedia.org/resource/Ammonium |
| Na | Sodium | http://dbpedia.org/resource/Sodium |
| K | Potassium | http://dbpedia.org/resource/Potassium |
| Ca | Calcium | http://dbpedia.org/resource/Calcium |
| Mg | Magnesium | http://dbpedia.org/resource/Magnesium |
| Cl | Chloride | http://dbpedia.org/resource/Chloride |
| NO3 | Nitrate | http://dbpedia.org/resource/Nitrate |
| SO4 | Sulfate | http://dbpedia.org/resource/Sulfate |
| Pb | Lead | http://dbpedia.org/resource/Lead |
| Cd | Cadmium | http://dbpedia.org/resource/Cadmium |
| NS_flag to Cd_flag | qualityFlag | http://dbpedia.org/ontology/qualityFlag |

### Table: measurement_variables

| Column | Concept | URI |
|--------|---------|-----|
| variable_code | identifier | http://dbpedia.org/ontology/identifier |
| label | name | http://dbpedia.org/ontology/name |
| unit | unit | http://dbpedia.org/ontology/unit |

Internal auto-generated ID columns (station_id PK, measurement_id, variable_id) are not mapped as they carry no scientific meaning.

In [1]:
from dbrepo.RestClient import RestClient

ENDPOINT    = "https://test.dbrepo.tuwien.ac.at"
USERNAME    = "e12551187@student.tuwien.ac.at"
PASSWORD    = "@Puthenpurayil1"
DATABASE_ID = "bfa4385b-54a9-4ae3-b4f4-cb503d7bb016"

client = RestClient(
    endpoint=ENDPOINT,
    username=USERNAME,
    password=PASSWORD
)

print("logged in as:", client.whoami())

e12551187@student.tuwien.ac.at
logged in as: e12551187@student.tuwien.ac.at


In [2]:
# table IDs from Owner A's notebook

TABLE_STATIONS      = "53688cc1-5205-4f25-af30-7feef2ea1b2b"
TABLE_PRECIPITATION = "d11966e6-f0a6-460d-900b-b56e627fc752"
TABLE_VARIABLES     = "7ed509f5-3356-4318-80b3-c6672b13c4b8"

In [3]:
# get column UUIDs from DBRepo

def get_column_ids(table_id):
    table = client.get_table(database_id=DATABASE_ID, table_id=table_id)
    return {col.name: col.id for col in table.columns}

stations_col_ids      = get_column_ids(TABLE_STATIONS)
precipitation_col_ids = get_column_ids(TABLE_PRECIPITATION)
variables_col_ids     = get_column_ids(TABLE_VARIABLES)

print("columns fetched successfully")

columns fetched successfully


In [4]:
# all URIs use DBpedia which is confirmed registered in the DBRepo test instance

DB  = "http://dbpedia.org/ontology/"
DBR = "http://dbpedia.org/resource/"

mappings = [

    # stations table
    (TABLE_STATIONS, stations_col_ids, "station_code", DB + "location"),
    (TABLE_STATIONS, stations_col_ids, "latitude",     DB + "latitude"),
    (TABLE_STATIONS, stations_col_ids, "longitude",    DB + "longitude"),

    # precipitation_measurements table
    (TABLE_PRECIPITATION, precipitation_col_ids, "sample_date", DB + "date"),
    (TABLE_PRECIPITATION, precipitation_col_ids, "station_id",  DB + "location"),
    (TABLE_PRECIPITATION, precipitation_col_ids, "NS",   DB  + "precipitation"),
    (TABLE_PRECIPITATION, precipitation_col_ids, "LF",   DB  + "electricalConductivity"),
    (TABLE_PRECIPITATION, precipitation_col_ids, "pH",   DB  + "pH"),
    (TABLE_PRECIPITATION, precipitation_col_ids, "NH4",  DBR + "Ammonium"),
    (TABLE_PRECIPITATION, precipitation_col_ids, "Na",   DBR + "Sodium"),
    (TABLE_PRECIPITATION, precipitation_col_ids, "K",    DBR + "Potassium"),
    (TABLE_PRECIPITATION, precipitation_col_ids, "Ca",   DBR + "Calcium"),
    (TABLE_PRECIPITATION, precipitation_col_ids, "Mg",   DBR + "Magnesium"),
    (TABLE_PRECIPITATION, precipitation_col_ids, "Cl",   DBR + "Chloride"),
    (TABLE_PRECIPITATION, precipitation_col_ids, "NO3",  DBR + "Nitrate"),
    (TABLE_PRECIPITATION, precipitation_col_ids, "SO4",  DBR + "Sulfate"),
    (TABLE_PRECIPITATION, precipitation_col_ids, "Pb",   DBR + "Lead"),
    (TABLE_PRECIPITATION, precipitation_col_ids, "Cd",   DBR + "Cadmium"),

    # quality flags, values: 1=valid, 4=contaminated, 7=missing
    (TABLE_PRECIPITATION, precipitation_col_ids, "NS_flag",  DB + "qualityFlag"),
    (TABLE_PRECIPITATION, precipitation_col_ids, "LF_flag",  DB + "qualityFlag"),
    (TABLE_PRECIPITATION, precipitation_col_ids, "pH_flag",  DB + "qualityFlag"),
    (TABLE_PRECIPITATION, precipitation_col_ids, "NH4_flag", DB + "qualityFlag"),
    (TABLE_PRECIPITATION, precipitation_col_ids, "Na_flag",  DB + "qualityFlag"),
    (TABLE_PRECIPITATION, precipitation_col_ids, "K_flag",   DB + "qualityFlag"),
    (TABLE_PRECIPITATION, precipitation_col_ids, "Ca_flag",  DB + "qualityFlag"),
    (TABLE_PRECIPITATION, precipitation_col_ids, "Mg_flag",  DB + "qualityFlag"),
    (TABLE_PRECIPITATION, precipitation_col_ids, "Cl_flag",  DB + "qualityFlag"),
    (TABLE_PRECIPITATION, precipitation_col_ids, "NO3_flag", DB + "qualityFlag"),
    (TABLE_PRECIPITATION, precipitation_col_ids, "SO4_flag", DB + "qualityFlag"),
    (TABLE_PRECIPITATION, precipitation_col_ids, "Pb_flag",  DB + "qualityFlag"),
    (TABLE_PRECIPITATION, precipitation_col_ids, "Cd_flag",  DB + "qualityFlag"),

    # measurement_variables table
    (TABLE_VARIABLES, variables_col_ids, "variable_code", DB + "identifier"),
    (TABLE_VARIABLES, variables_col_ids, "label",         DB + "name"),
    (TABLE_VARIABLES, variables_col_ids, "unit",          DB + "unit"),
]

print("total mappings:", len(mappings))

total mappings: 34


In [5]:
# push every mapping to DBRepo

ok = 0
failed = 0

for table_id, col_ids, column_name, uri in mappings:

    col_id = col_ids.get(column_name)
    if col_id is None:
        print(f"skip   {column_name:18s} (not found in table)")
        continue

    try:
        client.update_table_column(
            database_id=DATABASE_ID,
            table_id=table_id,
            column_id=col_id,
            concept_uri=uri
        )
        print(f"ok     {column_name:18s} {uri}")
        ok += 1
    except Exception as e:
        print(f"failed {column_name:18s} {str(e)[:80]}")
        failed += 1

print(f"\n{ok} ok, {failed} failed")

ok     station_code       http://dbpedia.org/ontology/location
ok     latitude           http://dbpedia.org/ontology/latitude
ok     longitude          http://dbpedia.org/ontology/longitude
ok     sample_date        http://dbpedia.org/ontology/date
ok     station_id         http://dbpedia.org/ontology/location
ok     NS                 http://dbpedia.org/ontology/precipitation
ok     LF                 http://dbpedia.org/ontology/electricalConductivity
ok     pH                 http://dbpedia.org/ontology/pH
ok     NH4                http://dbpedia.org/resource/Ammonium
ok     Na                 http://dbpedia.org/resource/Sodium
ok     K                  http://dbpedia.org/resource/Potassium
ok     Ca                 http://dbpedia.org/resource/Calcium
ok     Mg                 http://dbpedia.org/resource/Magnesium
ok     Cl                 http://dbpedia.org/resource/Chloride
ok     NO3                http://dbpedia.org/resource/Nitrate
ok     SO4                http://dbpedia.org/re

In [6]:
# verify the concepts are saved

for table_id, name in [(TABLE_STATIONS, "stations"), (TABLE_PRECIPITATION, "precipitation_measurements"), (TABLE_VARIABLES, "measurement_variables")]:
    print(f"\n{name}")
    table = client.get_table(database_id=DATABASE_ID, table_id=table_id)
    for col in table.columns:
        concept = getattr(col, "concept", None)
        uri = concept.uri if concept else "not set"
        print(f"  {col.name:20s} {uri}")


stations
  station_id           not set
  station_code         http://dbpedia.org/ontology/location
  latitude             http://dbpedia.org/ontology/latitude
  longitude            http://dbpedia.org/ontology/longitude

precipitation_measurements
  measurement_id       not set
  station_id           http://dbpedia.org/ontology/location
  sample_date          http://dbpedia.org/ontology/date
  NS                   http://dbpedia.org/ontology/precipitation
  NS_flag              http://dbpedia.org/ontology/qualityFlag
  LF                   http://dbpedia.org/ontology/electricalConductivity
  LF_flag              http://dbpedia.org/ontology/qualityFlag
  pH                   http://dbpedia.org/ontology/pH
  pH_flag              http://dbpedia.org/ontology/qualityFlag
  NH4                  http://dbpedia.org/resource/Ammonium
  NH4_flag             http://dbpedia.org/ontology/qualityFlag
  Na                   http://dbpedia.org/resource/Sodium
  Na_flag              http://dbpedia.or